In [9]:
import pandas as pd
sms_data = pd.read_csv('../data/raw/sms.csv')
calls_data = pd.read_csv('../data/raw/calls.csv')
bt_data = pd.read_csv('../data/raw/bt_symmetric.csv')
fb_data = pd.read_csv('../data/raw/fb_friends.csv').rename(columns={'# user_a': 'user_a'})

In [10]:
# Longest streak of consecutive days on which each pair exchanged at least one text
SECONDS_PER_DAY = 24 * 60 * 60

text_days = sms_data.loc[
    sms_data['sender'].ne(sms_data['recipient']),
    ['timestamp', 'sender', 'recipient'],
].copy()
text_days['user_a'] = text_days[['sender', 'recipient']].min(axis=1)
text_days['user_b'] = text_days[['sender', 'recipient']].max(axis=1)
text_days['day'] = text_days['timestamp'] // SECONDS_PER_DAY
text_days = text_days[['user_a', 'user_b', 'day']].drop_duplicates()

def longest_consecutive_streak(days):
    days = days.sort_values()
    streak_id = days.diff().ne(1).cumsum()
    return int(days.groupby(streak_id).size().max())

longest_text_streaks = (
    text_days.groupby(['user_a', 'user_b'])['day']
    .agg(longest_consecutive_text_streak_days=longest_consecutive_streak)
    .reset_index()
    .sort_values(
        'longest_consecutive_text_streak_days', ascending=False
    )
    .reset_index(drop=True)
)

longest_text_streaks

,user_a,user_b,longest_consecutive_text_streak_days
0,20,21,28
1,176,578,28
2,274,632,28
3,279,617,28
4,48,49,27
...,...,...,...
692,5,269,1
693,8,419,1
694,8,557,1
695,9,669,1


In [11]:
# Longest streak of consecutive days on which each pair had at least one call
call_days = calls_data.loc[
    calls_data['caller'].ne(calls_data['callee']),
    ['timestamp', 'caller', 'callee'],
].copy()
call_days['user_a'] = call_days[['caller', 'callee']].min(axis=1)
call_days['user_b'] = call_days[['caller', 'callee']].max(axis=1)
call_days['day'] = call_days['timestamp'] // SECONDS_PER_DAY
call_days = call_days[['user_a', 'user_b', 'day']].drop_duplicates()

longest_call_streaks = (
    call_days.groupby(['user_a', 'user_b'])['day']
    .agg(longest_consecutive_call_streak_days=longest_consecutive_streak)
    .reset_index()
    .sort_values(
        'longest_consecutive_call_streak_days', ascending=False
    )
    .reset_index(drop=True)
)

longest_call_streaks

,user_a,user_b,longest_consecutive_call_streak_days
0,289,578,28
1,47,449,9
2,560,736,7
3,634,681,7
4,90,91,7
...,...,...,...
616,92,262,1
617,634,677,1
618,615,691,1
619,611,706,1


In [12]:
# Sum the Facebook-friend counts of two users (shared friends count for both users)
facebook_neighbors = pd.concat(
    [
        fb_data[['user_a', 'user_b']].rename(
            columns={'user_a': 'user', 'user_b': 'friend'}
        ),
        fb_data[['user_b', 'user_a']].rename(
            columns={'user_b': 'user', 'user_a': 'friend'}
        ),
    ],
    ignore_index=True,
).query('user != friend').drop_duplicates()

facebook_friend_counts = facebook_neighbors.groupby('user')['friend'].nunique()

def total_facebook_friends(user_a, user_b):
    """Return friend_count(user_a) + friend_count(user_b)."""
    return int(
        facebook_friend_counts.get(user_a, 0)
        + facebook_friend_counts.get(user_b, 0)
    )

# Example: total_facebook_friends(0, 1)

In [13]:
# Total number of texts sent by either of two users to any recipient
def total_texts_sent(user_a, user_b):
    """Return the combined number of texts sent by the two users."""
    return int(sms_data['sender'].isin([user_a, user_b]).sum())

# Example: total_texts_sent(0, 1)

In [14]:
# Create every unordered pair of people and export their combined Facebook-friend count
from itertools import combinations

all_people = (
    pd.concat(
        [
            sms_data['sender'],
            sms_data['recipient'],
            calls_data['caller'],
            calls_data['callee'],
            bt_data['user_a'],
            bt_data['user_b'],
            fb_data['user_a'],
            fb_data['user_b'],
        ],
        ignore_index=True,
    )
    .dropna()
    .astype(int)
)
all_people = sorted(all_people[all_people >= 0].unique())

pair_facebook_friend_counts = pd.DataFrame(
    combinations(all_people, 2),
    columns=['user_a', 'user_b'],
)
pair_facebook_friend_counts['total_facebook_friend_count'] = (
    pair_facebook_friend_counts['user_a'].map(facebook_friend_counts).fillna(0)
    + pair_facebook_friend_counts['user_b'].map(facebook_friend_counts).fillna(0)
).astype(int)

pair_facebook_friend_counts.to_csv(
    '../data/interim/pair_facebook_friend_counts.csv', index=False
)

pair_facebook_friend_counts

,user_a,user_b,total_facebook_friend_count
0,0,1,25
1,0,2,26
2,0,3,37
3,0,4,49
4,0,5,45
...,...,...,...
351536,845,848,30
351537,845,850,39
351538,846,848,26
351539,846,850,35


In [15]:
# Create every unordered pair of people and export their combined outbound-text count
pair_total_texts_sent = pd.DataFrame(
    combinations(all_people, 2),
    columns=['user_a', 'user_b'],
)

texts_sent_counts = sms_data.groupby('sender').size()
pair_total_texts_sent['total_texts_sent'] = (
    pair_total_texts_sent['user_a'].map(texts_sent_counts).fillna(0)
    + pair_total_texts_sent['user_b'].map(texts_sent_counts).fillna(0)
).astype(int)

pair_total_texts_sent.to_csv(
    '../data/interim/pair_total_texts_sent.csv', index=False
)

pair_total_texts_sent

,user_a,user_b,total_texts_sent
0,0,1,64
1,0,2,62
2,0,3,209
3,0,4,144
4,0,5,83
...,...,...,...
351536,845,848,9
351537,845,850,9
351538,846,848,15
351539,846,850,15


In [16]:
# Export every unordered pair and its longest consecutive-day call streak
pair_longest_call_streaks = pd.DataFrame(
    combinations(all_people, 2),
    columns=['user_a', 'user_b'],
)

pair_longest_call_streaks = pair_longest_call_streaks.merge(
    longest_call_streaks,
    on=['user_a', 'user_b'],
    how='left',
    validate='one_to_one',
)
pair_longest_call_streaks['longest_consecutive_call_streak_days'] = (
    pair_longest_call_streaks['longest_consecutive_call_streak_days']
    .fillna(0)
    .astype(int)
)

pair_longest_call_streaks.to_csv(
    '../data/interim/pair_longest_call_streaks.csv', index=False
)

pair_longest_call_streaks

,user_a,user_b,longest_consecutive_call_streak_days
0,0,1,0
1,0,2,0
2,0,3,0
3,0,4,0
4,0,5,0
...,...,...,...
351536,845,848,0
351537,845,850,0
351538,846,848,0
351539,846,850,0


In [17]:
# Export every unordered pair and its longest consecutive text-day streak
pair_longest_text_streaks = pd.DataFrame(
    combinations(all_people, 2),
    columns=['user_a', 'user_b'],
)

pair_longest_text_streaks = pair_longest_text_streaks.merge(
    longest_text_streaks,
    on=['user_a', 'user_b'],
    how='left',
    validate='one_to_one',
)
pair_longest_text_streaks['longest_consecutive_text_streak_days'] = (
    pair_longest_text_streaks['longest_consecutive_text_streak_days']
    .fillna(0)
    .astype(int)
)

pair_longest_text_streaks.to_csv(
    '../data/interim/pair_longest_text_streaks.csv', index=False
)

pair_longest_text_streaks

,user_a,user_b,longest_consecutive_text_streak_days
0,0,1,0
1,0,2,0
2,0,3,0
3,0,4,0
4,0,5,0
...,...,...,...
351536,845,848,0
351537,845,850,0
351538,846,848,0
351539,846,850,0
